# GEMS — Geological feature discovery & fault association (gfd.1-r1)

Hypothesis-driven study: can a transformation of the GeoDAWN variables reveal mapped fault traces that the current models do not recover, and does it hold across geological environments?

* Formula language, operations and metrics come from the Layer Lab engine (`streamlit/gems_layer_lab/lab`), so features behave identically in the Streamlit studio, here and in exports.
* `mode='FULL'` = native 100 m full grid (final measurements, ~5–8 GB RAM); `mode='FAST'` = one window for exploration.
* Mapped faults enter evaluation only; unlabelled ≠ non-fault; held-out polarity is frozen on training-side regions of the repository's GEMS geographic folds; nothing here is an official DTI unless the canonical scorer section says so.
* Every section saves CSV / Parquet / PNG / SVG under `MyDrive/GEMS_2/outputs/geological_feature_discovery/` and a checkpoint; re-running reuses valid checkpoints.

## 00 · Configuration, Drive, repository (pinned commit)

In [ ]:
REF = 'main'                          # pin the pushed commit SHA for a reproducible run
CONFIG = {
    'gems_root': '/content/drive/MyDrive/GEMS_2',
    'features': 'data/raw/geodawn/gems-geodawn-numerical-features.tif',
    'faults': 'data/raw/faults/existing_faults.tif',
    'template': 'data/raw/example_submission.tif',
    'folds': 'outputs/benchmark/folds/geo_k3_b20km_buf13km_s2026/fold_map.tif',   # GEMS geographic folds
    'domains': 'outputs/geological_discovery_eda/rasters/spatial_domains/domains.npz',  # optional
    'w2_heldout': None,                # aligned held-out W2 score raster (+ <raster>.provenance.json) or None
    'mode': 'FULL',                    # FULL | FAST
    'fast_window': {'row': 1600, 'col': 1400, 'height': 600, 'width': 600},
    'budgets': [0.005, 0.01, 0.02, 0.05],
    'tiles_km': [1, 2, 5, 10, 20],
    'min_pos': 20, 'min_neg': 200,
    'complementarity_top_k': 10, 'unique_budget': 0.01,
    'n_boot': 200, 'block_px': 20, 'seed': 2026,
    'canonical_dti': False,            # canonical scorer on thresholded maps per fold (diagnostic only)
    'covariate': 'det_elev_slope',     # label-free covariate for matched controls
}

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = Path('/content/GEMS')
if not (CODE / '.git').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/mybenkhadda/GEMS---drivendata.git', str(CODE)], check=True)
subprocess.run(['git', '-C', str(CODE), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(CODE), 'checkout', '-q', ('origin/' + REF) if len(REF) < 40 else REF], check=True)
HEAD = subprocess.run(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
if 'gems_eda' in sys.modules:
    raise RuntimeError('gems_eda already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(CODE) + '[colab]', 'pytest', 'tabulate'], check=True)
sys.path.insert(0, str(CODE))
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
import gems_eda.feature_discovery as GFD
from gems_eda.feature_discovery import FeatureDiscoveryExperiment
CONFIG['commit'] = HEAD
E = FeatureDiscoveryExperiment(CONFIG)
OUT = E.store.root
def fig(name, width=900):
    p = OUT / 'figures' / f'{name}.png'
    display(Image(str(p), width=width)) if p.exists() else print('no figure', p)
print('commit', HEAD, '| code rev', GFD.FD_CODE_REV, '| outputs', OUT)

In [ ]:
# Optional: synthetic tests of the module and of the Layer Lab engine it reuses (no competition data)
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_feature_discovery.py'], cwd=str(CODE), capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])

## 01 · Load layers, fault labels, footprint, metadata (grid validated: 3730 × 3292, EPSG:32611, 100 m, 19 bands)

In [ ]:
REP = E.load()
display(pd.DataFrame([REP['checks']]))
display(E.tables['band_metadata'])

## 02 · Baseline raw-variable fault association (both polarities, exploratory)

In [ ]:
RAW = E.global_association(E.E.inputs, section='raw_fault_association')
display(RAW[['feature', 'auroc_high', 'auroc_low', 'ap_lift_high', 'ap_lift_low', 'recall_1pct_high', 'recall_1pct_low', 'n_valid', 'n_fault_px']])

## 03 · Magnetic, gravity, terrain, basement transformations (catalog)

`gems_eda/feature_discovery/catalog.py`: R01–R21 (Layer Lab program), R25–R29 magnetic hypotheses, R31–R42 additional hypotheses, `neg_rtp` reference (direction twin of `rtp`: no new information). Features are saved as NPY + provenance JSON under `layers/full/` and reused when their formula hash is unchanged.

In [ ]:
print(GFD.catalog.CATALOG[:1500], '...')

## 04 · Custom formula laboratory

Restricted language (AST interpreter, no `eval`/`exec`): assignments, arithmetic, approved functions (`lab.ops`), references to raw bands and catalog / earlier features, optional `PARAMS` (strict literal). Mapped faults and W2 maps cannot be inputs. Definitions are versioned in `formulas/custom_*.json`.

In [ ]:
CUSTOM_FEATURES = '''
mag_edge = gradmag(rtp, sigma=3)
mag_texture = local_std(rtp, 5)
my_mag_ratio = ratio(mag_edge, mag_texture, 0.05)
my_terrain_mag = normdiff(gradmag(det_elev_slope, 3), mag_edge, 0.05)
'''
E.register_formulas(CUSTOM_FEATURES)
E.compute_features()                      # catalog + custom; unchanged features are reused
E.plot_feature('my_mag_ratio'); fig('map_my_mag_ratio')
display(pd.DataFrame([E.evaluate_against_faults('my_mag_ratio')]))
display(E.compare_features(['rtp', 'R25_magnetic_edge', 'R26_magnetic_residual', 'my_mag_ratio']))

## 05 · Global fault-association benchmarking (both directions; exploratory)

In [ ]:
G = E.global_association()
display(G[['feature', 'family', 'auroc_high', 'auroc_low', 'ap_lift_high', 'ap_lift_low', 'cliffs_delta', 'median_difference', 'recall_1pct_high', 'recall_1pct_low', 'better_direction_EXPLORATORY']].head(30))
fig('auroc_ap_polarity'); fig('recall_vs_budget')
TOP = list(G[G.status == 'OK'].feature.head(15))
SEL = list(dict.fromkeys(TOP + ['neg_rtp', 'grav_broad', 'R20_basement_terrain_contrast', 'R25_magnetic_edge', 'R26_magnetic_residual', 'R27_magnetic_texture', 'R28_normalized_magnetic_edge', 'R29_magnetic_multiscale']))
print('features carried to sections 06-12:', SEL)

## 06 · Fault-centred spatial response (distance bands 0 m … > 5 km, fault-normal profiles, matched controls) — retrospective diagnostics

In [ ]:
D = E.fault_distance_responses(SEL)
display(E.load_table('fault_response_classes'))
fig('fault_distance_response'); fig('fault_normal_profiles')

## 07 · Local & regional association (3×3 regions, 1–20 km tiles, label-free domains; INSUFFICIENT_DATA cells never filled)

In [ ]:
R = E.regional_association(SEL, G)
display(E.load_table('feature_scale_comparison').sort_values(['partition', 'same_polarity_share'], ascending=[True, False]).head(40))
fig('region_feature_heatmap')
# local maps: np.load(OUT / 'datasets/regional_maps.npz') keys '<feature>__<partition>__<metric>'

## 08 · Localisation & top-budget recovery (exact, within 300 m, components, branches, off-trace area, localisation index)

In [ ]:
REC = E.fault_recovery(SEL)
display(REC[REC.budget == 0.01].sort_values('recall_300m', ascending=False).head(30))

## 09 · Complementarity & unique fault recovery (references: -RTP, broad gravity, R20; equal 1 % budget)

In [ ]:
U = E.unique_recovery(SEL)
display(U.sort_values('feature_only', ascending=False).head(30))
fig('unique_fault_recovery_map'); fig('complementarity_matrix'); fig('redundancy_matrix')

## 10 · Optional frozen-W2 failure analysis (only with a genuinely held-out W2 raster and provenance file)

In [ ]:
W = E.w2_failure_analysis(SEL)
if W is not None: display(W)

## 11 · Spatially held-out validation (GEMS geographic folds: polarity frozen on train side, scored on the held-out fold, 2 km block CIs; combinations only after singles)

In [ ]:
H = E.heldout_validation(SEL)
display(H[['fold', 'feature', 'status', 'direction', 'dev_auroc', 'eval_auroc', 'eval_auroc_ci_lo', 'eval_auroc_ci_hi', 'eval_recall_1pct', 'eval_recall_300m_1pct']])
display(E.load_table('heldout_selected_on_dev')); display(E.load_table('heldout_combinations'))
if CONFIG['canonical_dti']:
    display(E.canonical_dti(SEL[:8]))

## 12 · Feature ranking, selection (PROMOTE / EXPERIMENTAL / REDUNDANT / REJECT) and export

In [ ]:
REC_T = E.rank_and_classify()
display(REC_T[['feature', 'class', 'reason', 'score', 'heldout_auroc_mean', 'heldout_recall_1pct_mean', 'unique_component_share_vs_best_reference', 'regional_same_polarity_share', 'localization_index']])
PROMOTED = list(REC_T[REC_T['class'] == 'PROMOTE'].feature)
print('PROMOTE:', PROMOTED)
if PROMOTED and CONFIG['mode'] == 'FULL':
    print(E.export_features(PROMOTED))                 # native GeoTIFFs + provenance
    print(E.publish_to_layer_lab(PROMOTED))            # Layer Lab registry -> FeatureStore for P6 / M9

## 13 · Scientific synthesis and next modelling recommendations (generated from the saved tables)

In [ ]:
display(Markdown(E.synthesis()))

### Reload in a later session

Run 00 + 01, then any section: valid checkpoints are reused (`force=True` recomputes). Direct reads: `E.load_table('global_fault_association')`, `E.get('R20_basement_terrain_contrast')` (memory-mapped NPY), `formulas/frozen_catalog.py` (formulas with frozen normalisation statistics), `layers/geotiff/*.tif`.